In [3]:
# Libraries import karo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'notebook_connected'

# CSV se data load karo
df = pd.read_csv(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\data\reliance_featured_data.csv', 
                 index_col='Date', 
                 parse_dates=True)

print("Data loaded! ✅")
print("Shape:", df.shape)
df.head()

Data loaded! ✅
Shape: (1335, 30)


,Open,High,Low,Close,Volume,MA_7,MA_21,MA_50,MA_200,RSI,...,Year,Month,Day,Day_of_Week,Quarter,Daily_Return,ATR,OBV,Price_Range,Price_Change
Date,,,,,,,,,,,,,,,,,,,,,
2020-11-05,886.435462,889.156033,876.029205,886.458130,37205373,886.710754,951.977867,979.580208,773.579933,29.551765,...,2020,11,5,3,4,2.184822,30.258433,-96710392,13.126828,0.022668
2020-11-06,902.328214,924.999736,896.955016,920.080017,66674086,887.857282,947.441392,978.841566,774.752826,38.018435,...,2020,11,6,4,4,3.792834,29.879495,-30036306,28.044720,17.751803
2020-11-09,935.020567,938.149220,923.185991,929.851440,35838510,889.398943,943.495457,978.248025,775.971108,41.445749,...,2020,11,9,0,4,1.062019,29.628486,5802204,14.963229,-5.169126
2020-11-10,941.776738,947.671358,925.543925,945.200134,36934242,891.345459,940.202680,978.282937,777.267365,46.756818,...,2020,11,10,1,4,1.650661,28.156450,42736446,22.127433,3.423396
2020-11-11,947.217858,949.938429,896.932308,905.592834,56724780,899.102356,934.081351,977.466306,778.380123,42.014122,...,2020,11,11,2,4,-4.190361,30.598503,-13988334,53.006120,-41.625024


In [4]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


*TARGET HUMESHA CLOSE PRICE HI HOGA*

*STEP - 1*

In [5]:
# Target variable banao — kal ki Close price
df['Target'] = df['Close'].shift(-1)

# Last row drop karo — uska target exist nahi karta
df = df.dropna()

print("Target variable created! ✅")
print("Shape:", df.shape)
print("\nLast 3 rows — Close vs Target:")
print(df[['Close', 'Target']].tail(3))

Target variable created! ✅
Shape: (1334, 31)

Last 3 rows — Close vs Target:
                  Close       Target
Date                                
2026-03-27  1348.099976  1343.900024
2026-03-30  1343.900024  1369.199951
2026-04-01  1369.199951  1350.500000


💡 Ye Kyun Kiya?

shift(-1) matlab — Target column mein agli din ki price daaldo!
Socho aise — model aaj ka data dekh ke kal ki price predict karega!
Ye hai Supervised Learning — model ko answer pata hota hai training mein! 😊

*STEP - 2*

In [6]:
# features and target ko yahan hum alag kar rahe hai
# Features — X (input)
X = df.drop(columns=['Target'])

# Target — y (output)
y = df['Target']

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nX columns:", X.columns.tolist())

Features (X) shape: (1334, 30)
Target (y) shape: (1334,)

X columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'MA_7', 'MA_21', 'MA_50', 'MA_200', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Close_Lag1', 'Close_Lag2', 'Close_Lag3', 'Close_Lag5', 'Year', 'Month', 'Day', 'Day_of_Week', 'Quarter', 'Daily_Return', 'ATR', 'OBV', 'Price_Range', 'Price_Change']


*STEP - 3*

In [7]:
# Train/Test split — 80% training, 20% testing
# shuffle=False — kyunki time series data hai, order matter karta hai!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    shuffle=False
)

print("Train/Test Split complete! ✅")
print(f"Training data:  {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Testing data:   {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nTraining period: {X_train.index[0].date()} to {X_train.index[-1].date()}")
print(f"Testing period:  {X_test.index[0].date()} to {X_test.index[-1].date()}")

Train/Test Split complete! ✅
Training data:  1067 rows (80%)
Testing data:   267 rows (20%)

Training period: 2020-11-05 to 2025-02-25
Testing period:  2025-02-27 to 2026-04-01


*STEP - 4*

In [8]:
# Scaler banao
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Training data pe fit karo aur transform karo
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

# Testing data pe sirf transform karo — fit nahi!
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

print("Scaling complete! ✅")
print(f"\nX_train_scaled min: {X_train_scaled.min():.2f}")
print(f"X_train_scaled max: {X_train_scaled.max():.2f}")

Scaling complete! ✅

X_train_scaled min: 0.00
X_train_scaled max: 1.00


*STEP - 5*

In [11]:
import pickle
import os

# Models folder banao
os.makedirs(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\models', exist_ok=True)

# Scalers save karo
with open(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\models\scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)

with open(r'C:\Users\pc\Documents\DATA SCIENCE PROJECTS\Stock Market Price Prediction & Analysis System\models\scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

print("Scalers saved! ✅")
print("models folder mein scaler_X.pkl aur scaler_y.pkl save ho gaye!")

Scalers saved! ✅
models folder mein scaler_X.pkl aur scaler_y.pkl save ho gaye!


DAY 6 SUMMARY

# Day 6 - Data Preprocessing

## Libraries Used
- Pandas, NumPy, Scikit-learn (MinMaxScaler, train_test_split)

## Steps Completed

### Step 1 - Target Variable
- Kal ki Close price ko Target banaya
- shift(-1) use kiya — aaj ke features se kal ki price predict karni hai
- Last row drop ki — uska target exist nahi karta

### Step 2 - Features aur Target Split
- X (Features) = 30 columns — model ka input
- y (Target) = 1 column — model ka output
- Shape: X = (1334, 30), y = (1334,)

### Step 3 - Train/Test Split
- Training data: 1067 rows (80%) — Nov 2020 to Feb 2025
- Testing data:  267 rows (20%)  — Feb 2025 to Apr 2026
- shuffle=False — time series ka order maintain kiya

### Step 4 - Scaling (MinMaxScaler)
- Sab features 0 to 1 ke beech scale kiye
- Training pe fit_transform kiya
- Testing pe sirf transform kiya — cheating se bachne ke liye
- X_train_scaled: min=0.00, max=1.00

### Step 5 - Scalers Save kiye
- scaler_X.pkl — features ka scaler
- scaler_y.pkl — target ka scaler
- Location: models/ folder

## Key Concepts Seekhe
- Supervised Learning — model ko input aur output dono pata hote hain
- Train/Test Split — 80% padhna, 20% exam
- Scaling — sab numbers ko equal importance dene ke liye
- shuffle=False — time series mein order matter karta hai